In [1]:
import os
import geopandas as gpd
import osmnx as ox
import polars as pl
import duckdb

# buildings =ox.features_from_place('Budapest, Hungary', tags={'building': True})
# buildings.to_parquet('osm_data/buildings.parquet')
ox.settings.use_cache = True
ox.settings.log_console = True

#address_points = ox.features_from_place('Budapest, Hungary', tags={'addr:street': True})
#address_points.to_parquet('osm_data/address_points.parquet')


In [51]:
db = duckdb.connect("osm_data/buildings.duckdb")
db.execute("""install spatial; load spatial;""")

In [3]:
db.query("""
         CREATE TABLE IF NOT EXISTS buildings AS
         SELECT * FROM parquet_scan('osm_data/buildings.parquet')
         """)

In [4]:
db.query("""
         create table if not exists telepulesek as from 'processed/teleplist.csv'
         """)

In [5]:
db.query(
    """create table if not exists keruletek as select id, nev, nev[10:11] as kerulet from telepulesek where nev like '%kerület%'"""
)

In [6]:
db.query(
    """create or replace table address_points 
    as select *, ST_Transform(geometry, 'EPSG:4326', 'EPSG:23700') AS geom_eov
    from parquet_scan('osm_data/address_points.parquet') 
    where "addr:street" is not null and ST_GeometryType(geometry) = 'POINT' """
)

In [7]:
db.query("""
create or replace table ingatlanarak as (
    select
        dense_rank() over (order by telaz, kozter) as street_id,
        row_number() over (partition by telaz, kozter order by ev) as recnum,
        telaz,
        kozter,
        ev,
        szoras as rel_szoras,
        total_ar,
        total_db,
        keruletek.nev as kerulet_nev,
        kerulet
    from 'processed/hu-ingatlan.csv'
    join keruletek on telaz = keruletek.id
)
""")

In [8]:
db.execute(
    """create or replace table buildings_subset as 
    (select
      buildings.id as osm_id,
      building,  
      geometry,
      ST_Transform(geometry, 'EPSG:4326', 'EPSG:23700') as geom_eov,
      ST_Buffer(ST_Transform(geometry, 'EPSG:4326', 'EPSG:23700'), 10) as geom_buffer,
      "addr:postcode" as postcode, 
      "addr:city" as city, 
      "addr:street" as street, 
      "addr:housenumber" as housenumber,
      "addr:postcode"[2:3] as kerulet,
      keruletek.nev as kerulet_nev,
      keruletek.id as telaz,
      ingatlanarak.street_id
    from buildings
    left join keruletek on keruletek.kerulet = buildings."addr:postcode"[2:3]
    left join ingatlanarak on ingatlanarak.telaz = keruletek.id and ingatlanarak.kozter = buildings."addr:street"
    where building is not null
      and ST_GeometryType(geometry) = 'POLYGON' 
       OR ST_GeometryType(geometry) = 'MULTIPOLYGON'
    )"""
)

In [9]:
db.execute("""
create or replace table address_candidates as
select
    b.osm_id,
    b.geometry as geometry,
    b.building,
    a.id,
    a."addr:postcode" as postcode,
    a."addr:city" as city, 
    a."addr:street" as street,
    a."addr:housenumber" as housenumber,
    a.geometry as address_geom,
    ST_Distance(b.geom_eov, a.geom_eov) as dist_m,
    row_number() over (
        partition by b.osm_id
        order by
            case when a.entrance == 'main' then 0 else 1 end,
            case when ST_Intersects(b.geom_eov, a.geom_eov) then 0 else 1 end,
            case when ST_Intersects(b.geom_buffer, a.geom_eov) then 0 else 1 end,
            ST_Distance(b.geom_eov, a.geom_eov),
            a.id
    ) as rn,
     keruletek.nev as kerulet_nev,
        keruletek.id as telaz,
    ingatlanarak.street_id 
from (from buildings_subset where street_id is null) b
left join address_points a
  --on ST_Intersects(b.geom_eov, a.geom_eov)
  on ST_Intersects(b.geom_buffer, a.geom_eov)
left join keruletek on keruletek.kerulet = a."addr:postcode"[2:3]
left join ingatlanarak on ingatlanarak.telaz = keruletek.id and ingatlanarak.kozter = a."addr:street";
""")

db.execute("""
create or replace table building_addresses as
select *
from address_candidates
where rn = 1;
""")

In [ ]:
# union the two table
db.execute(
    """
    create or replace table building_addresses_union as (
    select osm_id, geometry, building, postcode, city, street, housenumber, kerulet_nev, telaz, street_id, 'point' as source
    from building_addresses
    where street_id is not null
    union
    select osm_id, geometry, building,  postcode, city, street, housenumber, kerulet_nev, telaz, street_id, 'building' as source
    from buildings_subset
    where street_id is not null
    )
    """
)

# Save to files

In [32]:
db.execute(
    """copy building_addresses_union to 'processed/building_addresses_union.gpkg' 
    WITH (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'buildings')"""
)

In [3]:
db.execute(
    """copy building_addresses_union to 'processed/building_addresses_union.csv' 
    """
)

In [33]:
db.execute("""
copy ingatlanarak to 'processed/bp_ingatlanarak.csv' with (format csv, header true);
           """)

In [39]:
db.execute(
    """
    copy (
    select i.street_id, bp.telaz, bp.kozter, bp.geometry from 'processed/bp-merged.csv' bp
    left join ingatlanarak i on i.telaz = bp.telaz and i.kozter = bp.kozter
    ) to 'processed/bp_ingatlanarak_with_street_id.gpkg' 
    WITH (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'streets')
""")


# Join rest with QGIS

In [16]:
# export to QGIS for nearest matching
db.execute(
    """copy (select
    osm_id, geometry, building, osm_id, postcode, city, street, housenumber, kerulet_nev, telaz, street_id, null as source
    from buildings_subset 
    where osm_id not in (select osm_id from building_addresses_union)
    )
    to 'processed/buildings_no_street_id.gpkg' 
    WITH (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'buildings')"""
)

In [61]:
db.execute("""
select st_astext(geom)  from 'qgis/buildings_20m_joined_wgs.gpkg'""").df()

,st_astext(geom)
0,MULTIPOLYGON (((19.20649358577233 47.436700248...
1,MULTIPOLYGON (((19.20685628296545 47.436730952...
2,MULTIPOLYGON (((19.207075982829778 47.43667625...
3,MULTIPOLYGON (((19.207104485602084 47.43653845...
4,MULTIPOLYGON (((19.206365924840583 47.43490534...
...,...
408310,MULTIPOLYGON (((19.029033628612456 47.45147739...
408311,MULTIPOLYGON (((19.02844372355207 47.451880185...
408312,MULTIPOLYGON (((19.028344420706926 47.45204138...
408313,MULTIPOLYGON (((19.028294917648573 47.45219858...


In [64]:
# import result
# import result
db.query("""
        create or replace table qgis_result as
         select distinct
           osm_id,
           geom as geometry,
           building,
           joined_street_id as street_id,
           'qgis' as source
         from 'qgis/buildings_20m_joined_wgs.gpkg' q
         join ingatlanarak i on i.street_id = q.joined_street_id
         where distance is not null and joined_street_id is not null;
        """)


In [65]:
db.execute("""
 select *, st_astext(geometry), st_isvalid(geometry), st_astext(st_centroid(geometry)) from qgis_result
""").df()

,osm_id,geometry,building,street_id,source,st_astext(geometry),st_isvalid(geometry),st_astext(st_centroid(geometry))
0,1137895879,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",yes,2997,qgis,MULTIPOLYGON (((19.203253161222342 47.43875091...,True,POINT (19.20329770109535 47.43876233815329)
1,1137895886,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",yes,2942,qgis,MULTIPOLYGON (((19.204303647641538 47.43909512...,True,POINT (19.20427226255292 47.43915772855199)
2,1137896503,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",yes,2942,qgis,MULTIPOLYGON (((19.205355712318248 47.44045733...,True,POINT (19.20526034935359 47.44044741500627)
3,1137897152,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",yes,3010,qgis,MULTIPOLYGON (((19.214055864449403 47.44028241...,True,POINT (19.21411534390168 47.44035551184985)
4,1137897968,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",yes,2942,qgis,MULTIPOLYGON (((19.206092404661803 47.44061124...,True,POINT (19.20604087314261 47.44049041471953)
...,...,...,...,...,...,...,...,...
56658,711364625,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",yes,1247,qgis,MULTIPOLYGON (((19.12367276720094 47.488700386...,True,POINT (19.123808947897636 47.4887860249579)
56659,711962067,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",yes,1227,qgis,MULTIPOLYGON (((19.126303375405733 47.48758341...,True,POINT (19.126121643553635 47.48747652389206)
56660,711966385,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",yes,1227,qgis,MULTIPOLYGON (((19.12583264928447 47.488935507...,True,POINT (19.125671054009103 47.48901574313638)
56661,713496964,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ...",garages,1166,qgis,MULTIPOLYGON (((19.001106976445065 47.41017905...,True,POINT (19.000683658602256 47.40996483283946)


In [66]:
db.execute("""
create or replace table final_building_addresses as
select 
                   osm_id, geometry, building, street_id, source
                   
                        from building_addresses_union
                        where ST_IsValid(geometry) 
union
select 
                   osm_id, geometry, building, street_id, source
                   
                        from qgis_result
                        where ST_IsValid(geometry)
                         
        """)

In [67]:
db.execute("""
           copy final_building_addresses to 'final/final_building_addresses.csv'
           """)

# Join to ingatlanarak

In [46]:
db.execute("""
    COPY (from buildings_subset where "addr:street" is not null) TO 'budapest_buildings_with_street.gpkg' 
    WITH (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'buildings');
""")
db.execute("""
    COPY (from buildings_subset where "addr:street" is null) TO 'budapest_buildings_without_street.gpkg' 
    WITH (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'buildings');
""")

BinderException: Binder Error: Referenced column "addr:street" not found in FROM clause!
Candidate bindings: "postcode"

LINE 2:     COPY (from buildings_subset where "addr:street" is not null) TO 'budapest_buildings_with_street...
                                              ^

In [47]:
db.execute("""PRAGMA table_info('buildings');""").fetchdf()

,cid,name,type,notnull,dflt_value,pk
0,0,geometry,GEOMETRY,False,None,False
1,1,access,VARCHAR,False,None,False
2,2,amenity,VARCHAR,False,None,False
3,3,building,VARCHAR,False,None,False
4,4,charge,VARCHAR,False,None,False
...,...,...,...,...,...,...
877,877,date:religious_title,VARCHAR,False,None,False
878,878,religious_title,VARCHAR,False,None,False
879,879,wheelchair:description:de,VARCHAR,False,None,False
880,880,element,VARCHAR,False,None,False


In [48]:
db.execute(
    """
        copy (
            select bp.telaz, bp.kozter, ST_GeomFromText(geometry) as geometry, i.street_id
            from 'processed/bp-merged.csv' bp  
            join ingatlanarak i on bp.telaz = i.telaz and bp.kozter = i.kozter
            where bp.telaz is not null and bp.kozter is not null
            ) to 'bp-merged_with_street_id.gpkg'
            WITH (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'streets');
        """
)

In [49]:
db.close()